# T5Gemma 2 — CEFR classification (fine-tuning, val)

Fine-tunes T5Gemma 2 on Write & Improve (A2–C1+, 8 levels), nominal
cross-entropy head only. Imports the encoder + head from `model.py` and
`evaluate_predictions` from `metrics.py` — the same module the ModernBERT
and Gemma 4 notebooks use, so all conditions are scored identically.

Ends by writing `dev_predictions.csv` in the same column layout as the
Gemma 4 and ModernBERT notebooks (`p_pred`, per-class probabilities, margin,
entropy, expected level), so the results notebook can read all four
conditions the same way.

In [1]:
import os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [2]:
# %%
import os, json, csv
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer, Trainer, TrainingArguments, DataCollatorWithPadding,
)

from model import T5Gemma2OrdinalClassifier   # encoder + mean-pool + head (from model.py)
from metrics import evaluate_predictions, LABEL_NAMES

LEVELS = ["A2", "A2+", "B1", "B1+", "B2", "B2+", "C1", "C1+"]   # matches Gemma 4
LABEL2ID = {lvl: i for i, lvl in enumerate(LEVELS)}
NUM_CLASSES = len(LEVELS)


@dataclass
class CFG:
    model_id: str = "google/t5gemma-2-4b-4b"
    train_path: str = "data/train.jsonl"
    eval_path: str  = "data/val.jsonl"
    text_col: str = "text"
    label_col: str = "label"
    max_length: int = 1024
    epochs: float = 4.0
    batch_size: int = 8
    grad_accum: int = 1
    lr: float = 2e-5
    weight_decay: float = 0.01
    bf16: bool = True
    use_lora: bool = True                       # flip on if VRAM is tight
    lora_r: int = 16
    output_dir: str = "runs/t5gemma2_nominal"
    seed: int = 42

cfg = CFG()

W0913 11:41:39.453000 35632 site-packages\torch\utils\_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0913 11:41:39.495000 35632 site-packages\torch\utils\_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


## Data

In [3]:
def read_rows(path):
    ext = os.path.splitext(path)[1].lower()
    if ext == ".jsonl":
        with open(path, encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    yield json.loads(line)
    elif ext == ".json":
        with open(path, encoding="utf-8") as f:
            yield from json.load(f)
    else:  # csv / tsv
        delim = "\t" if ext == ".tsv" else ","
        with open(path, encoding="utf-8", newline="") as f:
            yield from csv.DictReader(f, delimiter=delim)


class CEFRDataset(Dataset):
    def __init__(self, path, tokenizer, cfg):
        self.ex, skipped = [], 0
        for row in read_rows(path):
            lab = str(row[cfg.label_col]).strip().upper()
            if lab not in LABEL2ID:          # drop A1/C2 or stray labels
                skipped += 1
                continue
            self.ex.append((str(row[cfg.text_col]), LABEL2ID[lab]))
        print(f"{path}: kept {len(self.ex)}, skipped {skipped} out-of-scope")
        self.tok, self.max_length = tokenizer, cfg.max_length

    def __len__(self):
        return len(self.ex)

    def __getitem__(self, i):
        text, label = self.ex[i]
        enc = self.tok(text, truncation=True, max_length=self.max_length)
        enc["labels"] = label
        return enc

In [4]:
tokenizer = AutoTokenizer.from_pretrained(cfg.model_id)
train_ds = CEFRDataset(cfg.train_path, tokenizer, cfg)
eval_ds  = CEFRDataset(cfg.eval_path,  tokenizer, cfg)
collator = DataCollatorWithPadding(tokenizer)

data/train.jsonl: kept 3797, skipped 0 out-of-scope
data/val.jsonl: kept 599, skipped 0 out-of-scope


In [5]:
# --- model ---
dtype = torch.bfloat16 if cfg.bf16 else torch.float32
model = T5Gemma2OrdinalClassifier(cfg.model_id, num_classes=NUM_CLASSES,
                                  mode="nominal", dtype=dtype)

if cfg.use_lora:
    from peft import LoraConfig, get_peft_model
    model = get_peft_model(model, LoraConfig(
        r=cfg.lora_r, lora_alpha=2 * cfg.lora_r, lora_dropout=0.05,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        modules_to_save=["head"],   # always fully train the small head
    ))
    model.print_trainable_parameters()

Loading weights:   0%|          | 0/1327 [00:00<?, ?it/s]

trainable params: 11,919,368 || all params: 4,311,858,048 || trainable%: 0.2764


In [6]:
# --- trainer ---
class NominalTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        labels = inputs.pop("labels")
        out = model(**inputs)
        loss = torch.nn.functional.cross_entropy(out.logits, labels.long())
        return (loss, out) if return_outputs else loss


def compute_metrics(eval_pred):
    logits = eval_pred.predictions
    if isinstance(logits, tuple):
        logits = logits[0]
    preds = np.asarray(logits).argmax(-1)
    return evaluate_predictions(eval_pred.label_ids, preds)   # identical to Gemma 4/ModernBERT

In [7]:
steps_per_epoch = len(train_ds) // (cfg.batch_size * cfg.grad_accum)
total_steps = int(steps_per_epoch * cfg.epochs)
warmup_steps = int(0.06 * total_steps)

# --- train ---
args = TrainingArguments(
    output_dir=cfg.output_dir,
    num_train_epochs=cfg.epochs,
    per_device_train_batch_size=cfg.batch_size,
    per_device_eval_batch_size=cfg.batch_size,
    gradient_accumulation_steps=cfg.grad_accum,
    learning_rate=cfg.lr,
    weight_decay=cfg.weight_decay,
    warmup_steps=warmup_steps,
    bf16=cfg.bf16,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="eval_qwk",
    greater_is_better=True,
    save_total_limit=2,
    seed=cfg.seed,
    report_to="none",
)

trainer = NominalTrainer(
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=eval_ds,
    data_collator=collator, compute_metrics=compute_metrics,
)
trainer.train()

c:\Users\coope\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\nn\modules\module.py:1369: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:40.)
  return t.to(


Epoch,Training Loss,Validation Loss,Qwk,Mae,Accuracy,Adjacent Accuracy,Precision Macro,Recall Macro,F1 Macro,Precision Weighted,Recall Weighted,F1 Weighted
1,1.102369,1.158356,0.862381,0.585977,0.494157,0.923205,0.436695,0.381030,0.335375,0.483541,0.494157,0.458208
2,0.952648,1.005303,0.889165,0.474124,0.565943,0.963272,0.547343,0.444618,0.436233,0.589409,0.565943,0.548950
3,0.967050,0.952957,0.905059,0.455760,0.574290,0.973289,0.507242,0.462155,0.454006,0.573540,0.574290,0.554209
4,0.897537,0.952110,0.902383,0.455760,0.577629,0.969950,0.635505,0.479193,0.492527,0.600396,0.577629,0.567187


TrainOutput(global_step=1900, training_loss=1.0918065000835218, metrics={'train_runtime': 8382.4627, 'train_samples_per_second': 1.812, 'train_steps_per_second': 0.227, 'total_flos': 0.0, 'train_loss': 1.0918065000835218, 'epoch': 4.0})

## Val evaluation — metrics + predictions

In [8]:
dev_out = trainer.predict(eval_ds, metric_key_prefix="eval")
dev_logits = dev_out.predictions
if isinstance(dev_logits, tuple):
    dev_logits = dev_logits[0]
dev_logits_t = torch.as_tensor(np.asarray(dev_logits, dtype=np.float32))
dev_preds = dev_logits_t.argmax(-1).numpy()
dev_gold = dev_out.label_ids

probs = torch.softmax(dev_logits_t, dim=-1).numpy()

print("best dev:", json.dumps(dev_out.metrics, indent=2))
print("logits shape:", tuple(dev_logits_t.shape))
print("prob rows sum to 1:", np.allclose(probs.sum(1), 1.0, atol=1e-4))

os.makedirs(cfg.output_dir, exist_ok=True)
with open(os.path.join(cfg.output_dir, "dev_metrics.json"), "w") as f:
    json.dump(dev_out.metrics, f, indent=2)

best dev: {
  "eval_loss": 0.9529570937156677,
  "eval_qwk": 0.9050588650532138,
  "eval_mae": 0.4557595993322204,
  "eval_accuracy": 0.5742904841402338,
  "eval_adjacent_accuracy": 0.9732888146911519,
  "eval_precision_macro": 0.5072422678414894,
  "eval_recall_macro": 0.4621547571872891,
  "eval_f1_macro": 0.45400602302584553,
  "eval_precision_weighted": 0.5735397497838998,
  "eval_recall_weighted": 0.5742904841402338,
  "eval_f1_weighted": 0.5542085805661472,
  "eval_runtime": 60.272,
  "eval_samples_per_second": 9.938,
  "eval_steps_per_second": 1.244
}
logits shape: (599, 8)
prob rows sum to 1: True


## Confusion matrix

In [9]:
import plotly.graph_objects as go
from sklearn.metrics import confusion_matrix

counts = confusion_matrix(dev_gold, dev_preds, labels=list(range(NUM_CLASSES)))
row_sums = counts.sum(axis=1, keepdims=True)
with np.errstate(divide="ignore", invalid="ignore"):
    norm = np.nan_to_num(np.divide(counts, row_sums, where=row_sums != 0))

annot = np.empty_like(counts, dtype=object)
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        annot[i, j] = f"{norm[i, j] * 100:.1f}%<br>({counts[i, j]})"

fig = go.Figure(go.Heatmap(
    z=norm, x=LEVELS, y=LEVELS,
    text=annot, texttemplate="%{text}",
    hoverongaps=False,
    colorscale="Greys", zmin=0, zmax=1,
    colorbar=dict(title="Row-normalized"),
))
fig.update_xaxes(title_text="Predicted")
fig.update_yaxes(title_text="True", autorange="reversed")
fig.update_layout(
    title=dict(text="T5Gemma 2 nominal — dev confusion matrix"),
    font=dict(family="Arial", size=16, color="black"),
    width=620, height=560,
    margin=dict(l=80, r=100, t=100, b=80),
)
fig.show()

## Predictions CSV

Same column layout as the other three conditions (Gemma 4 prompted, Gemma 4
LoRA, ModernBERT nominal) — unlike the earlier CORN-vs-nominal notebook,
this includes `p_pred` and the per-class probabilities so the results
notebook can compute ECE/AUROC without touching this file again.

In [10]:
srt = np.sort(probs, axis=1)[:, ::-1]
rows = []
for i, (g, p) in enumerate(zip(dev_gold, dev_preds)):
    g, p = int(g), int(p)
    lo, hi = max(0, p - 1), min(NUM_CLASSES - 1, p + 1)
    rows.append(dict(
        gold=g, pred=p, gold_label=LEVELS[g], pred_label=LEVELS[p],
        correct=int(g == p), adjacent=int(abs(g - p) <= 1),
        p_pred=float(probs[i, p]),
        p_adjacent=float(probs[i, lo:hi + 1].sum()),
        margin=float(srt[i, 0] - srt[i, 1]),
        entropy=float(-(probs[i] * np.log(probs[i] + 1e-12)).sum()),
        exp_level=float((probs[i] * np.arange(NUM_CLASSES)).sum()),
        **{f"p_{lvl}": float(probs[i, j]) for j, lvl in enumerate(LEVELS)},
    ))

dev_df = pd.DataFrame(rows)
dev_df.to_csv(os.path.join(cfg.output_dir, "dev_predictions.csv"), index=False)

print(f"Saved to {cfg.output_dir}/dev_metrics.json and dev_predictions.csv")

Saved to runs/t5gemma2_nominal/dev_metrics.json and dev_predictions.csv


## Does confidence separate correct from wrong?
### 0.5 is chance, >0.7 is what I want to see

In [11]:
from sklearn.metrics import roc_auc_score


def ece(d, n_bins=10):
    b = pd.cut(d["p_pred"], np.linspace(0, 1, n_bins + 1))
    g = d.groupby(b, observed=True)
    gap = (g["correct"].mean() - g["p_pred"].mean()).abs()
    return float((gap * g.size()).sum() / len(d))


print(f"mean p_pred  {dev_df['p_pred'].mean():.3f}")
print(f"mean entropy {dev_df['entropy'].mean():.3f}  (max {np.log(NUM_CLASSES):.3f})")
print(f"ECE          {ece(dev_df):.4f}")
print()
for col, s in [("p_pred", dev_df["p_pred"]), ("margin", dev_df["margin"]),
               ("entropy", -dev_df["entropy"]), ("p_adjacent", dev_df["p_adjacent"])]:
    print(f"{col:12s} exact {roc_auc_score(dev_df['correct'], s):.3f}   "
          f"adjacent {roc_auc_score(dev_df['adjacent'], s):.3f}")
print()
print(dev_df.groupby("correct")[["p_pred", "margin", "entropy"]].mean().round(3))

mean p_pred  0.632
mean entropy 0.864  (max 2.079)
ECE          0.0590

p_pred       exact 0.622   adjacent 0.672
margin       exact 0.627   adjacent 0.651
entropy      exact 0.581   adjacent 0.655
p_adjacent   exact 0.593   adjacent 0.687

         p_pred  margin  entropy
correct                         
0         0.603   0.304    0.896
1         0.654   0.399    0.841


In [12]:
edges = np.arange(0.0, 1.01, 0.1)
BANDS = list(zip(edges[:-1], edges[1:],
                 [f"{lo:.1f}\u2013{hi:.1f}" for lo, hi in zip(edges[:-1], edges[1:])]))

n = len(dev_df)
band_rows = []
for lo, hi, name in reversed(BANDS):
    s = dev_df[(dev_df["p_pred"] >= lo) & (dev_df["p_pred"] < hi)]
    band_rows.append(dict(
        conf=name, preds=len(s), share=f"{len(s)/n*100:.0f}%",
        acc=round(s["correct"].mean(), 2) if len(s) else None,
        adj=round(s["adjacent"].mean(), 2) if len(s) else None,
    ))

print(pd.DataFrame(band_rows).to_string(index=False))

   conf  preds share  acc  adj
0.9–1.0      3    1% 1.00 1.00
0.8–0.9     52    9% 0.71 1.00
0.7–0.8    136   23% 0.71 0.99
0.6–0.7    165   28% 0.55 0.98
0.5–0.6    143   24% 0.53 0.97
0.4–0.5     94   16% 0.43 0.96
0.3–0.4      6    1% 0.17 0.83
0.2–0.3      0    0%  NaN  NaN
0.1–0.2      0    0%  NaN  NaN
0.0–0.1      0    0%  NaN  NaN
